In [23]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

df = pd.read_csv("air_fryers_clean_brand_year.csv")

df["log_share"] = np.log(df["brand_share"])

df.head()

,category,year,brand,purchase_count,product_count,avg_price,avg_rating,compact_share,dual_basket_share,oven_style_share,rotisserie_share,window_share,market_purchases,brand_share,log_brand_share,log_share
0,air_fryers,2019,chefman,1146,10,72.963695,4.434119,1.000000,0.0,0.780977,0.243455,0.184119,15076,0.076015,-2.576826,-2.576826
1,air_fryers,2019,cosori,11,2,159.990000,4.581818,1.000000,0.0,0.090909,0.090909,0.000000,15076,0.000730,-7.222964,-7.222964
2,air_fryers,2019,cuisinart,1616,22,229.465274,4.481312,0.993812,0.0,0.889851,0.000000,0.000000,15076,0.107190,-2.233150,-2.233150
3,air_fryers,2019,dash,3011,19,55.176333,4.390767,1.000000,0.0,0.973431,0.000000,0.000000,15076,0.199721,-1.610832,-1.610832
4,air_fryers,2019,gowise usa,4405,45,83.575551,4.552259,0.999773,0.0,0.129398,0.128490,0.000000,15076,0.292186,-1.230364,-1.230364


In [24]:
exclude_cols = [
    "brand",
    "year",
    "brand_share",
    "log_brand_share",
    "avg_price",
    "avg_rating",
    "log_share",

    "purchase_count",
    "market_purchases"
]

feature_cols = [col for col in df.columns if col not in exclude_cols]

feature_cols = df[feature_cols].select_dtypes(include="number").columns.tolist()

print(feature_cols)

['product_count', 'compact_share', 'dual_basket_share', 'oven_style_share', 'rotisserie_share', 'window_share']


In [25]:
brand_dummies = pd.get_dummies(df["brand"], drop_first=True)
year_dummies = pd.get_dummies(df["year"], drop_first=True)

brand_dummies.head()
year_dummies.head()

,2020,2021,2022,2023
0,False,False,False,False
1,False,False,False,False
2,False,False,False,False
3,False,False,False,False
4,False,False,False,False


In [26]:
X = pd.concat([
    df[["avg_price", "avg_rating"]],
    df[feature_cols],
    brand_dummies,
    year_dummies
], axis=1)

X.columns = X.columns.astype(str)

y = df["log_share"]

X.head()

,avg_price,avg_rating,product_count,compact_share,dual_basket_share,oven_style_share,rotisserie_share,window_share,cosori,cuisinart,...,gowise usa,instant_pot,ninja,nuwave,oster,ultrean,2020,2021,2022,2023
0,72.963695,4.434119,10,1.000000,0.0,0.780977,0.243455,0.184119,False,False,...,False,False,False,False,False,False,False,False,False,False
1,159.990000,4.581818,2,1.000000,0.0,0.090909,0.090909,0.000000,True,False,...,False,False,False,False,False,False,False,False,False,False
2,229.465274,4.481312,22,0.993812,0.0,0.889851,0.000000,0.000000,False,True,...,False,False,False,False,False,False,False,False,False,False
3,55.176333,4.390767,19,1.000000,0.0,0.973431,0.000000,0.000000,False,False,...,False,False,False,False,False,False,False,False,False,False
4,83.575551,4.552259,45,0.999773,0.0,0.129398,0.128490,0.000000,False,False,...,True,False,False,False,False,False,False,False,False,False


In [27]:
model = LinearRegression()
model.fit(X, y)

results = pd.DataFrame({
    "variable": ["intercept"] + list(X.columns),
    "coefficient": [model.intercept_] + list(model.coef_)
})

results

,variable,coefficient
0,intercept,-18.691775
1,avg_price,-0.042029
2,avg_rating,2.405390
3,product_count,0.039143
4,compact_share,6.445432
5,dual_basket_share,0.449013
6,oven_style_share,0.703472
7,rotisserie_share,-1.778652
8,window_share,8.825698
9,cosori,1.832768


In [28]:
r2 = model.score(X, y)
print("R^2:", r2)

R^2: 0.8088294557107853


#### Answers
1. The estimated price coefficient is approximately −0.042. In other words, a one-unit increase in price (with other factors constant) produces 0.042 decrease in log market share.
2. Yes, it is negative, which is important because it matches normal demand behavior: when price goes up, demand should go down.
3. The product features associated with higher demand are window_share (8.825), followed by compact_share (6.445), oven_style_share (0.7035), and dual_basket_share (0.4490), meaning that window_share and compact_share are highly positively associated with market share demand. In contrast, the rotisserie_share (-1.7786) has a negative coefficient, indicating that there is lower demand for rotisserie-style air fryers. 
4. The largest brand dummy coefficients are Cuisinart (6.8399) Oster (5.209), Ninja (3.859), and Nuwave (3.5877), indicating higher demand for products from these brands. 
5. The **most negative** coefficient for years is 2022 (-0.504), indicating a high drop in demand for that year in comparison to the baseline year of 2019. However, all of the year coefficients are negative, showing that in general there was a lower demand for air fryers in other years compared to 2019. 
6. The $R^2$ was 0.8088, meaning that the model explains 80.88% of the variation in log market share. 